# Volume 07 — The AutoML Toolkit
### PyCaret, FLAML, AutoGluon, H2O AutoML & Auto-sklearn — Let the Machine Search for You

*Consolidated from: Grok Chapter 21, ChatGPT Chapters 29–33 (PyCaret, H2O AutoML, Auto-sklearn, FLAML, AutoGluon)*

---

## Why This Volume Matters — and Its Honest Limits

Volume 05 and 06 taught you to build and tune models yourself. **AutoML** automates the *entire* remaining workflow — trying dozens of algorithms, engineering basic preprocessing, and tuning hyperparameters — often producing a strong baseline model from raw data in minutes rather than the hours or days manual experimentation would take. For a time-constrained business analyst who needs a credible model fast, this is a genuine force multiplier.

⚠️ **The one sentence to remember from this entire volume, repeated in nearly every source notebook because it matters that much: AutoML is a force multiplier, not a replacement for domain knowledge.** It will not catch a fundamentally wrong target variable, a leaked feature (a column that accidentally contains information from the future), or a business framing that doesn't match the actual decision being made. Every AutoML result in this volume still needs the judgment you built in Volumes 01–06 to sanity-check it before it drives a real decision.

**Installation reality check:** several of these libraries (H2O, Auto-sklearn especially) have heavy, sometimes finicky installation requirements and platform restrictions (Auto-sklearn, for instance, doesn't support native Windows). The workflow code below is accurate and runnable in a properly set-up environment, but don't be surprised if getting one of these installed takes real troubleshooting — that's a known, common experience, not a sign you've done something wrong.

### A Shared Dataset for This Volume

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

np.random.seed(42)
n = 3000
df = pd.DataFrame({
    "tenure": np.random.randint(1, 72, n),
    "monthly_charges": np.round(np.random.uniform(20, 120, n), 2),
    "contract": np.random.choice(["Month-to-month", "One year", "Two year"], n, p=[0.55, 0.25, 0.2]),
    "internet": np.random.choice(["DSL", "Fiber", "None"], n),
    "tech_support": np.random.choice(["Yes", "No"], n, p=[0.3, 0.7]),
    "senior": np.random.binomial(1, 0.16, n),
    "tickets": np.random.poisson(1.3, n)
})
logit = (-1.7 + 0.018*df.monthly_charges - 0.03*df.tenure
         + 0.85*(df.contract == "Month-to-month").astype(int)
         + 0.4*(df.tech_support == "No").astype(int) + 0.3*df.tickets + 0.35*df.senior)
df["churn"] = (np.random.rand(n) < 1/(1+np.exp(-logit))).astype(int)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["churn"])

---